<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/cosyvoice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/코딩공부/project_folder

/content/drive/MyDrive/코딩공부/project_folder


In [ ]:
!git clone https://github.com/FunAudioLLM/CosyVoice.git

Cloning into 'CosyVoice'...
remote: Enumerating objects: 2276, done.
remote: Counting objects: 100% (681/681), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 2276 (delta 571), reused 496 (delta 496), pack-reused 1595 (from 2)
Receiving objects: 100% (2276/2276), 1.60 MiB | 7.49 MiB/s, done.
Resolving deltas: 100% (1420/1420), done.


### 코드 분석

In [ ]:
import torch
from torch import nn, sin, pow
from torch.nn import Parameters

import math
from typing import Tuple
from torch import nn

In [ ]:
class Swish(torch.nn.Module):
  def forward(self,x: torch.Tensor)-> torch.Tensor:
    return x*torch.sigmoid(x)

class Snake(nn.Module):
  def __init__(self,in_feature,alpha=1.0 alpha_trainable=True,alpha_logscale=False):

    super(Snake,self).__init__()
    self.in_features=in_features

    self.alpha_logscale=alpha_logscale
    if self.alpha_logscale:
      self.alpha=Parameter(torch.zeros(in_features)*alpha)

    else:
      self.alpha=Parameter(torch.ones(in_features)*alpha)

    self.alpha.requires_grad=alpha_trainable

    self.no_div_by_zero=0.000000001


  def forward(self,x):
    alpha=self.alpha.unsqueeze(0).unsqueeze(-1)
    if self.alpha_logscale:
      alpha=torch.exp(alpha)
    x=x+(1.0/(alpha+self.no_div_by_zero))*pow(sin(x*alpha),2)

    return x

In [ ]:
# 멀티헤드 어텐션 구현

class MultiHeadAttention(nn.Module):
  def __init__(self,
               n_head:int,
               n_feat:int,
               dropout_rate:float,
               key_bias:bool=True):

    super().__init__()
    assert n_feat%n_head==0

    self.d_k=n_feat//n_head
    self.h=n_head
    self.linear_q=nn.Linear(n_feat,n_feat)
    self.linear_k=nn.Linear(n_feat,n_feat,bias=key_bias)
    self.linear_v=nn.Linear(n_feat,n_feat)
    self.linear_out=nn.Linear(n_feat,n_feat)
    self.dropout=nn.Dropout(p=dropout_rate)

    def forward_qkv(
        self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
      n_batch=query.size(0)
      q=self.linear_q(query).view(n_batch,-1,self.h,self.d_k)
      k=self.linear_k(key).view(n_batch,-1,self.h,self.d_k)
      v=self.linear_v(value).view(n_batch,-1,self.h,self.d_k)

      q=q.transpose(1,2)
      k=k.transpose(1,2)
      v=v.transpose(1,2)

      return q,k,v


    def forward_attention(
        self,
        value=torch.Tensor,
        scores: torch.Tensor,
        mask:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool)
    )->torch.Tensor:
      n_batch=value.size(0)
      if mask.size(2)>0:
        mask=mask.unsqueeze(1).eq(0)
        mask=mask[:,:,:,:scores.size(-1)]
        scores=scores.masked_fill(mask,-float('inf'))
        attn=torch.softmax(scores,dim=-1).masked_fill(mask,0.0)

      else:
        attn=torch.softmax(scores,dim=-1)

      p_attn=self.dropout(attn)
      x=torch.matmul(p_attn,value)
      x=(x.transpose(1,2).contiguous().view(n_batch,-1,self.h*self.d_k))

      return self.linear_out(x)

  def forward(
      self,
      query:torch.Tensor,
      key:torch.Tensor,
      value:torch.Tensor,
      mask:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      pos_emb:torch.Tensor=torch.empty(0),
      cache:torch.Tensor=torch.zeros((0,0,0,0))
  )->Tuple[torch.Tensor,torch.Tensor]:

    q,k,v=self.forward_qkv(query,key,value)
    if cache.size(0)>0:
      key_cache, value_cache=torch.split(cache,
                                         cache.size(-1)//2,
                                         dim=-1)
      k=torch.cat([key_cache,k],dim=2)
      v=torch.cat([value_cache,v],dim=2)
    new_cache=torch.cat((k,v),dim=-1)

    scores=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.d_k)
    return self.forward_attention(v,scores,mask),new_cache





In [ ]:
class RelPositionMultiHeadAttention(MultiHeadttention):
  def __init__(self,
               n_head:int,
               n_feat:int,
               dropout_rate:float,
               key_bias:bool=True):
    super().__init__(n_head,n_feat,dropout_rate,key_bias)
    self.linear_pos=nn.Linear(n_feat,n_feat,bias=False)

    self.pos_bias_u=nn.Parameter(torch.Tensor(self.h,self.d_k))
    self.pos_bias_v=nn.Parameter(torch.Tensor(self.h,self.d_k))
    torch.nn.init.xavier_uniform_(self.pos_bias_u)
    torch.nn.init.xavier_uniform_(self.pos_bias_v)

  def rel_shift(self,x:torch.Tensor)->torch.Tensor:
    zero_pad=torch.zeros((x.size()[0],x.size()[1],x.size()[2],1),
                         device=x.device,
                         dtype=x.dtype)
    x_padded=torch.cat([zero_pad,x],dim=-1)
    x_padded=x_padded.view(x.size()[0],
                           x.size()[1],
                           x.size(3)+1,x.size(2))
    x=x_padded[:,:,1:].view_as(x)[
        :,:,:,: x.size(-1)//2+1
    ]
    return x

  def forward(
      self,
      query: torch.Tensor,
      key:torch.Tensor,
      value:torch.Tensor,
      mask:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      pos_emb:torch.Tensor=torch.empty(0),
      cache:torch.Tensor=torch.zeros((0,0,0,0))
  )->Tuple[torch.Tensor,torch.Tensor]

    q,k,v=self.forward_qkv(query,key,value)
    q=q.transpose(1,2)

    if cache.size(0)>0:
      key_cache,value_cache=torch.split(cache,cache.size(-1)//2,dim=-1)
      k=torch.cat([key_cache,k],dim=2)
      v=torch.cat([value_cache,v],dim=2)

    new_cache=torch.cat((k,v),dim=-1)

    n_bach_pos=pos_emb.size(0)
    p=self.linear_pos(pos_emb).view(n_batch_pos,-1,self.h,self.d_k)

    q_with_bias_u=(q+self.pos_bias_u).transpose(1,2)
    q_with_bias_v=(q+self.pos_bias_v).transpose(1,2)

    matrix_ac=torch.matmul(q_with_bias_u,k.transpose(-2,-1))

    matrix_bd=torch.matmul(q_with_bias_v,p.transpose(-2,-1))

    if matrix_ac.shape!=matrix_bd.shape:
      matrix_bd=self.rel_shift(matrix_bd)

    scores=(matrix_ac+matrix_bd)/math.sqrt(self.d_k)

    return self.forward_attention(v,scores,mask),new_cache

In [ ]:
class ConvolutionModule(nn.Module):

  def __init__(self,
               channels:int,
               kernel_size:int=15,
               activation:nn.Module=nn.ReLU(),
               norm:str="batch_norm",
               casual:bool=False,
               bias:bool=True):
    super().__init__()

    self.pointwise_conv1=nn.Conv1d(
        channels,
        2*channels,
        kernel_size=1,
        stride=1,
        padding=0,
        bias=bias,
    )

    if casual:
      padding=0
      self.lorder=kernel_size-1
    else:
      assert(kernel_size-1)%2==0
      padding=(kernel_size-1)//2
      self.lorder=0

    self.depthwise_conv=Conv1d(
        channels,
        channels,
        kernel_size,
        stride=1,
        padding=padding,
        groups=channels,
        bias=bias,
    )

    assert norm in ['batch_norm','layer_norm']
    if norm=="batch_norm":
      self.use_layer_norm=False
      self.norm=nn.BatchNorm1d(channels)
    else:
      self.use_layer_norm=True
      self.norm=nn.LayerNorm(channels)

    self.pointwise_conv2=nn.Conv1d(
        channels,
        channels,
        kernel_size=1,
        stride=1,
        padding=0,
        bias=bias,
    )
    self.activation=activation


  def forward(
      self,
      x:torch.Tensor,
      mask_pad:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      cache:torch.Tensor=torch.zeros((0,0,0)),
  )->Tuple[torch.Tensor,torch.Tensor]:

    x=x.transpose(1,2)

    if mask_pad.size(2)>0:
      x.masked_fill_(-mask_pad,0.0)

    if self.lorder>0:
      if cache.size(2)==0:
        x=nn.functional.pad(x,(self.lorder,0),'constant',0.0)
      else:
        assert cache.size(0)==x.size(0)
        assert cache.size(1)==x.size(1)
        x=torch.cat((cache,x),dim=2)
      assert (x.size(2)>self.lorder)
      new_cache=x[:,:,-self.lorder:]
    else:
      new_cache=torch.zeros((0,0,0),dtype=x.dtype,device=x.device)


    x=self.pointwise_conv1(x)
    x=nn.functional.glu(x,dim=1)

    x=self.depthwise_conv(x)
    if self.use_layer_norm:
      x=x.transpose(1,2)
    x=self.activation(self.norm(x))
    if self.use_layer_norm:
      x=x.transpose(1,2)
      x=self.pointwise_conv2(x)

    if mask_pad.size(2)>0:
      x.masked_fill_(-mask_pad,0.0)

    return x.transpose(1,2),new_cache

In [ ]:
class DecoderLayer(nn.Module):

  def __init__(
      self,
      size_attn:nn.Module,
      src_attn:Optional[nn.Module],
      feed_forward:nn.Module,
      dropout_rate:float,
      normalize_before:bool=True,
  ):

  super().__init__()
  self.size=size
  self.self_attn=self_attn
  self.src_attn=src_attn
  self.feed_forward=feed_forward
  self.norm1=nn.LayerNorm(size,eps=1e-5)
  self.norm2=nn.LayerNorm(size,eps=1e-5)
  self.norm3=nn.LayerNorm(size,eps=1e-5)
  self.dropout=nn.Dropout(dropout_rate)
  self.normalize_before=normalize_before


  def forward(
      self
      tgt:torch.Tensor,
      tgt_mask:torch.Tensor,
      memory:torch.Tensor,
      memory_mask:torch.Tensor,
      cache:Optional[torch.Tensor]=None,
  ) -> Tuple[torch.Tensor,torch.Tensor,torch.Tensor,torch.Tensor]:

    residual=tgt
    if self.normalize_before:
      tgt=self.norm(tgt)

    if cache is None:
      tgt_q=tgt
      tgt_q_mask=tgt_mask

    else:
      assert cache.shape==(
          tgt.shape[0],
          tgt.shape[1]-1,
          self.size,
      ),"{cache.shape} == {(tgt.shape[0], tgt.shape[1] - 1, self.size)}"
      tgt_q = tgt[:, -1:, :]
      residual=residual[:,-1:,:]
      tgt_q_mask=tgt_mask[:,-1:,:]

    x=residual+self.dropout(
        self.self_attn(tgt_q,tgt,tgt,tgt_q_mask)[0])
    if no self.normalize_before:
      x=self.norm1(x)

    if self.src_attn is not None:
      residual=x
      if self.normalize_before:
        x=self.norm2(x)
      x=residual+self.dropout(
          self.src_attn(x,memory,memory,memory_mask)[0])
      if not self.normalize_before:
        x=self.norm2(x)

    residual=x
    if self.normalize_before:
      x=self.norm3(x)
    x=residual+self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x=self.norm3(x)

    if cache is not None:
      x=torch.cat([cache,x],dim=1)

    return x, tgt_mask,memory,memory_mask

In [ ]:
class PositionalEncoding(torch.nn.Module):
  def __init__(self,
               d_model:int,
               dropout_rate:float,
               max_len:int=5000,
               reverse:bool=False):

    super().__init__()
    self.d_model=d_model
    self.xscale=math.sqrt(self.d_model)
    self.dropout=torch.nn.Dropout(p=dropout_rate)
    self.max_len=max_len

    self.pe=torch.zeros(self.max_len,self.d_model)
    position=torch.arange(0,self.max_len,dtype=torch.float32).unsqueeze(1)

    div_term=torch.exp(torch.arange(0,self.d_model,2,dtype=torch.float32)* -(math.log(10000.0)/self.d_model))
    self.pe[:,0::2]=torch.sin(position*div_term)
    self.pe[:,1::2]=torch.cos(position*div_term)
    self.pe=self.pe.unsqueeze(0)


  def forward(self,
              x:torch.Tensor,
              offset:Union[int,torch.Tensor]=0) -> Tuple[torch.Tensor,torch.Tensor]:

    self.pe=self.pe.to(x.device)
    pos_emb=self.position_encoding(offset,x.size(1),Fasle)
    x=x*self.xscale+pos_emb
    return self.dropout(x),self.dropout(pos_emb)

  def position_encoding(self,
                        offset:Union[int,torch.Tensor],
                        size:int,
                        apply_dropout:bool=True) -> torch.Tensor:

    if isinstance(offset,int):
      assert offset+size<=self.max_len
      pos_emb=self.pe[:,offset:offset+size]

    elif isinstance(ofset,torch.Tensor) and offset.dim()==0:
      assert offset+size<=self.max_len
      pos_emb=self.pe[:,offset:offset+size]

    else:
      assert torch.max(offset)+size<=self.max_len
      index=offset.unsqueeze(1)+torch.arange(0,size).to(offset.device)
      flag=index>0
      flag=index*flag
      pos_emb=F.embedding(index,self.pe[0])

    if apply_dropout:
      pos_emb=self.dropout(pos_emb)
    return pos_emb




In [ ]:
class RelPositionalEncoding(PositionalEncoding):
  def __init__(self,d_model,int,dropout_rate: float,max_len:int=5000):
    super().__init__(d_model,dropout_rate,max_len,reverse=True)

  def forward(self,
              x:torch.Tensor,
              offset:Union[int,torch.Tensor]=0)->Tuple[torch.Tensor,torch.Tensor]:
    self.pe=self.pe.to(x.device)
    x=x*self.xscale
    pos_emb=self.position_encoding(offset,x.size(1),False)
    return self.dropout(x),self.dropout(pos_emb)

In [ ]:
class WhisperPositionalEncoding(PositionalEncoding):
  def __init__(self,d_model:int, dropout_rate:float,max_len:int=5000):
    super().__init__(d_model,dropout_rate,max_len)
    self.xscale=1.0
    log_timescale_increment=np.log(10000)/(d_model//2-1)
    inv_timescales=torch.exp(-log_timescale_increment*torch.arange(d_model//2))
    scaled_time=torch.arange(max_len)[:,np.newaxis]*inv_timescales[np.nexaxis,:]
    pe=torch.cat([torch.sin(scaled_time),torch.cos(scaled_tiem)],dim=1)
    delattr(self,"pe")
    self.register_buffer("pe",pe.unsqueeze(0))

In [ ]:
class LearnablePositionalEncoding(PositionalEncoding):
  def __init__(self,d_model: int, dropout_rate:float, max_len:int=448):
    super().__init__(d_model,dropout_rate,max_len)
    self.pe=torch.nn.Parameter(torch.empty(1,max_len,d_model))
    self.xscale=1.0

class NoPositionalEncoding(torch.nn.Module):
  def __init__(self,d_model:int, dropout_rate:float):
    super().__init__()
    self.d_model=d_model
    self.dropout=torch.nn.Dropout(p=dropout_rate)

  def forward(self,
              x:torch.Tensor,
              offset:Union[int,torch.Tensor]=0) -> Tuple[torch.Tensor,torch.Tensor]:

    pos_emb=torch.zeros(1,x.size(1),self.d_model).to(x.device)
    return self.dropout(x),pos_emb

  def position_encoding(self,offset:Union[int,torch.Tensor],
                        size:int)->torch.Tensor:
    return torch.zeros(1,size,self.d_model)


class EspnetRelPositionalEncoding(torch.nn.Module):
  def __init__(self,d_model:int,dropout_rate:float,max_len:int=5000):
    super(EspnetRelPositionalEncoding,self).__init__()
    self.d_model=d_model
    self.xscale=math.sqrt(self.d_model)
    self.dropout=torch.nn.Dropout(p=dropout_rate)
    self.pe=None
    self.extend_pe(torch.tensor(0.0).expand(1,max_len))

  def extend_pe(self,x:torch.Tensor):
    if self.pe is not None:
      if self.pe.size(1)>=x.size(1)*2-1:
        if self.pe.dtype!=x.dtype or self.pe.device!=x.device:
          self.pe=self.pe.to(dtype=x.dtype,device=x.device)
        return

    pe_positive=torch.zeros(x.size(1),self.d_model)
    pe_negative=torch.zeros(x.size(1),self.d_model)
    position=torch.arange(0,x.size(1),dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
            torch.arange(0, self.d_model, 2, dtype=torch.float32)
            * -(math.log(10000.0) / self.d_model)
        )

    pe_positive[:, 0::2] = torch.sin(position * div_term)
    pe_positive[:, 1::2] = torch.cos(position * div_term)
    pe_negative[:, 0::2] = torch.sin(-1 * position * div_term)
    pe_negative[:, 1::2] = torch.cos(-1 * position * div_term)

    pe_positive=torch.flip(pe_positive,[0]).unsqueeze(0)
    pe_negative=pe_negative.unsqueeze(0)
    pe=torch.cat([pe_positive,pe_neagtive],dim=1)
    self.pe=pe.to(device=x.device,dtype=x.dtype)

  def forward(self,x:torch.Tensor,offset:Union[int,torch.Tensor]=0) -> Tuple[torch.Tensor,torch.Tensor]:
    self.extend_pe(x)
    x=x*self.xscale
    pos_emb=self.position_encoding(size=x.size(1),offset=offset)
    return self.dropout(x),self.dropout(pos_emb)

  def position_encoding(self,
                      offset: Union[int, torch.Tensor],
                      size: int) -> torch.Tensor:
    """ For getting encoding in a streaming fashion

    Attention!!!!!
    we apply dropout only once at the whole utterance level in a none
    streaming way, but will call this function several times with
    increasing input size in a streaming scenario, so the dropout will
    be applied several times.

    Args:
        offset (int or torch.tensor): start offset
        size (int): required size of position encoding

    Returns:
        torch.Tensor: Corresponding encoding
    """
    # How to subscript a Union type:
    #   https://github.com/pytorch/pytorch/issues/69434
    if isinstance(offset, int):
      pos_emb = self.pe[
          :,
          self.pe.size(1) // 2 - size - offset + 1: self.pe.size(1) // 2 + size + offset,
      ]
    elif isinstance(offset, torch.Tensor):
      pos_emb = self.pe[
          :,
          self.pe.size(1) // 2 - size - offset + 1: self.pe.size(1) // 2 + size + offset,
      ]
    return pos_emb

In [ ]:
class TransformerEncoderLayer(nn.Module):
  def __init__(
      self,
      size:int,
      self_attn:torch.nn.Module,
      feed_forward:torch.nn.Module,
      feed_forward:torch.nn.Module,
      dropout_rate:float,
      normalize_before:bool=True,
  ):
    super().__init__()
    self.self_attn=self_attn
    self.feed_forward=feed_forward
    self.norm1=nn.LayerNorm(size,eps=1e-5)
    self.norm2=nn.LayerNorm(size,eps=1e-5)
    self.dropout=nn.Dropout(dropout_rate)
    self.size=size
    self.normalize_before=normalize_before

  def forward(
      self,
      x:torch.Tensor,
      mask:torch.Tensor,
      pos_emb=torch.Tensor,
      mask_pad:torch.Tensor=torch.ones((0,0,0),dtype=torch.bool),
      att_cache:torch.Tensor=torch.zeros((0,0,0,0)),
      cnn_cache:torch.Tensor=torch.zeros((0,0,0,0)),
      )->Tuple[torch.Tensor, torch.Tensor,torch.Tensor,torch.Tensor]:

    residual=x
    if slef.normalize_before:
      x=self.norm1(x)
    x_att,new_att_cache=self.self_attn(x,x,x,mask,pos_emb=pos_emb,cache=att_cache)
    x=residual+self.dropout(x_att)
    if not self.normalize_before:
      x=self.norm1(x)

    residual=x
    if self.normalize_before:
      x=self.norm2(x)
    x=residual+self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x=self.norm2(x)

    fake_cnn_cache=torch.zeros((0,0,0),dtype=x.dtype,device=x.device)
    return x,mask,new_att_cache,fake_cnn_cache


class ConformerEncoderLayer(nn.Module):
 def __init__(
     self,
     size:int,
     self_attn:torch.nn.Module,
     feed_forward:Optional[nn.Module]=None,
     feed_forward_macaron:Optional[nn.Module]=None,
     conv_module:Optional[nn.Module]=None,
     dropout_rate:float=0.1,
     normalize_before:bool=True,
    ):

    super().__init__()
    self.self_attn=self.attn
    self.feed_forward=feed_forward
    self.feed_forward_macaron=feed_forward_macaron
    self.conv_module=conv_module
    self.norm_ff=nn.LayerNorm(size,eps=1e-12)
    self.norm_mha=nn.LayerNorm(size,eps=1e-12)
    if feed_forward is not None:
      self.norm_ff_macaron=nn.LayerNorm(size,eps=1e-12)
      self.ff_scale=0.5
    else:
      self.ff_scale=1.0

    if self.conv_module is not None:
      self.norm_conv=nn.LayerNorm(size,eps=1e-12)
      self.norm_final=nn.LayerNorm(
          size, eps=1e-12)
    self.dropout=nn.Dropout(dropout_rate)
    self.size=size
    self.normalize_before=normalize_before


  def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor,
        pos_emb: torch.Tensor,
        mask_pad: torch.Tensor = torch.ones((0, 0, 0), dtype=torch.bool),
        att_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
        cnn_cache: torch.Tensor = torch.zeros((0, 0, 0, 0)),
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:


    # whether to use macaron style
    if self.feed_forward_macaron is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_ff_macaron(x)
      x = residual + self.ff_scale * self.dropout(
        self.feed_forward_macaron(x))
      if not self.normalize_before:
        x = self.norm_ff_macaron(x)

    # multi-headed self-attention module
    residual = x
    if self.normalize_before:
      x = self.norm_mha(x)
    x_att, new_att_cache = self.self_attn(x, x, x, mask, pos_emb,
                                          att_cache)
    x = residual + self.dropout(x_att)
    if not self.normalize_before:
      x = self.norm_mha(x)

    # convolution module
    # Fake new cnn cache here, and then change it in conv_module
    new_cnn_cache = torch.zeros((0, 0, 0), dtype=x.dtype, device=x.device)
    if self.conv_module is not None:
      residual = x
      if self.normalize_before:
        x = self.norm_conv(x)
      x, new_cnn_cache = self.conv_module(x, mask_pad, cnn_cache)
      x = residual + self.dropout(x)

      if not self.normalize_before:
        x = self.norm_conv(x)

    # feed forward module
    residual = x
    if self.normalize_before:
      x = self.norm_ff(x)

    x = residual + self.ff_scale * self.dropout(self.feed_forward(x))
    if not self.normalize_before:
      x = self.norm_ff(x)

    if self.conv_module is not None:
      x = self.norm_final(x)

    return x, mask, new_att_cache, new_cnn_cache

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 83)

In [ ]:
class LabelSmoothingLoss(nn.Module):
  def __init__(self,
               size:int,
               padding_idx:int,
               smoothing:float,
               normalize_length:bool=False):
    super(LabelSmoothingLoss,self).__init__()
    self.criterion=nn.KLDivLoss(reduction="none")
    self.padding_idx=padding_idx
    self.confidence=1.0-smoothing
    self.smoothing=smoothing
    self.size=size
    self.normalize_length=normalize_length

  def forward(self,x:torch.Tensor,target:torch.Tensor)->torch.Tensor:
    assert x.size(2)==self.size
    batch_size=x.size(0)
    x=x.view(-1,self.size)
    target=target.view(-1)
    true_dist=torch.zeros_like(x)
    true_dixt.fill_(self.smoothing/(self.size-1))
    ignore=target==self.padding_idx
    total=len(target)==self.padding_idx
    target=target.masked_fill(ignore,0)
    true_dist.scatter_(1,target.unsqueeze(1),self.confidence)
    kl=self.criterion(torch.log_softmax(x,dim=1),true_dist)
    denom=total if self.normalize_length else batch_size
    return kl.masked_fill(ignore.unsqueeze(1),0).sum/denom

In [ ]:
class PositionwiseFeedForward(torch.nn.Module):
  def __init__(
      self,
      idim:int,
      hidden_units:int,
      dropout_rate:float
      activation:torch.nn.Module=torch.nn.ReLU(),
  )
    super(PositionalwiseFeedForward,self).__init_()
    self.w_1=torch.nn.Linear(idim,hidden_units)
    self.activation=activation
    self.dropout=torch.nn.Dropout(dropout_rate)
    self.w_2=torch.nn.Linear(hidden_units,idim)

  def forward(self,xs:torch.Tensor)->torch.Tensor:
    return self.w_2(self.dropout(self.activation(self.w_1(xs))))


class MoEFFNLayer(torch.nn.Module):
  def __init__(
      self,
      n_expert:int,
      n_expert_per_token:int,
      idim:int,
      hidden_units:int,
      dropout_rate=float,
      activation:torch.nn.Module=torch.nn.ReLU(),
  ):
    super(MoEFFNLayer,self).__init__()
    self.experts = torch.nn.ModuleList(
            PositionwiseFeedForward(idim, hidden_units, dropout_rate,
                                    activation) for _ in range(n_expert))
    self.n_expert_per_token=n_expert_per_token

  def forward(self,xs:torch.Tensor)->torch.Tensor:
    B,L,D=xs.size()
    xs=xs.view(-1,D)
    router=self.gate(xs)
    logits,indices=torch.topk(router,self.n_expert_per_token)
    weights = torch.nn.functional.softmax(
            logits, dim=1,
            dtype=torch.float).to(dtype=xs.dtype)
    output=torch.zeros_like(xs)
    for i, expert in enumerate(self.experts):
      mask=indices==i
      batch_idx,ith_expert=torch.where(mask)
      output[batch_idx]+=weights[batch_idx,ith_expert,None]*expert(xs[batach_idx])
    return output.view(B,L,D)

In [ ]:
# Copyright (c) 2021 Mobvoi Inc (Binbin Zhang, Di Wu)
#               2024 Alibaba Inc (Xiang Lyu)
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# Modified from ESPnet(https://github.com/espnet/espnet)
"""Subsampling layer definition."""

from typing import Tuple, Union

import torch


class BaseSubsampling(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.right_context = 0
        self.subsampling_rate = 1

    def position_encoding(self, offset: Union[int, torch.Tensor],
                          size: int) -> torch.Tensor:
        return self.pos_enc.position_encoding(offset, size)


class EmbedinigNoSubsampling(BaseSubsampling):
    """Embedding input without subsampling
    """

    def __init__(self, idim: int, odim: int, dropout_rate: float,
                 pos_enc_class: torch.nn.Module):
        super().__init__()
        self.embed = torch.nn.Embedding(idim, odim)
        self.pos_enc = pos_enc_class

    def forward(
        self,
        x: torch.Tensor,
        x_mask: torch.Tensor,
        offset: Union[int, torch.Tensor] = 0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
        x = self.embed(x)
        x, pos_emb = self.pos_enc(x, offset)
        return x, pos_emb, x_mask


class LinearNoSubsampling(BaseSubsampling):
    """Linear transform the input without subsampling

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

    def __init__(self, idim: int, odim: int, dropout_rate: float,
                 pos_enc_class: torch.nn.Module):
        """Construct an linear object."""
        super().__init__()
        self.out = torch.nn.Sequential(
            torch.nn.Linear(idim, odim),
            torch.nn.LayerNorm(odim, eps=1e-5),
            torch.nn.Dropout(dropout_rate),
        )
        self.pos_enc = pos_enc_class
        self.right_context = 0
        self.subsampling_rate = 1

    def forward(
        self,
        x: torch.Tensor,
        x_mask: torch.Tensor,
        offset: Union[int, torch.Tensor] = 0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
        x = self.out(x)
        x, pos_emb = self.pos_enc(x, offset)
        return x, pos_emb, x_mask


class Conv1dSubsampling2(BaseSubsampling):
    """Convolutional 1D subsampling (to 1/2 length).
       It is designed for Whisper, ref:
       https://github.com/openai/whisper/blob/main/whisper/model.py

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

    def __init__(self, idim: int, odim: int, dropout_rate: float,
                 pos_enc_class: torch.nn.Module):
        """Construct an Conv1dSubsampling2 object."""
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv1d(idim, odim, kernel_size=3, padding=1),
            torch.nn.GELU(),
            torch.nn.Conv1d(odim, odim, kernel_size=3, stride=2, padding=1),
            torch.nn.GELU(),
        )
        self.pos_enc = pos_enc_class
        # The right context for every conv layer is computed by:
        # (kernel_size - 1) * frame_rate_of_this_layer
        self.subsampling_rate = 2
        # 4 = (3 - 1) * 1 + (3 - 1) * 1
        self.right_context = 4

    def forward(
        self,
        x: torch.Tensor,
        x_mask: torch.Tensor,
        offset: Union[int, torch.Tensor] = 0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 2.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 2.
            torch.Tensor: positional encoding

        """
        time = x.size(1)
        x = x.transpose(1, 2)  # (b, f, t)
        x = self.conv(x)
        x = x.transpose(1, 2)  # (b, t, f)
        x, pos_emb = self.pos_enc(x, offset)
        return x, pos_emb, x_mask[:, :, (time + 1) % 2::2]


class Conv2dSubsampling4(BaseSubsampling):
    """Convolutional 2D subsampling (to 1/4 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

    def __init__(self, idim: int, odim: int, dropout_rate: float,
                 pos_enc_class: torch.nn.Module):
        """Construct an Conv2dSubsampling4 object."""
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv2d(1, odim, 3, 2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(odim, odim, 3, 2),
            torch.nn.ReLU(),
        )
        self.out = torch.nn.Sequential(
            torch.nn.Linear(odim * (((idim - 1) // 2 - 1) // 2), odim))
        self.pos_enc = pos_enc_class
        # The right context for every conv layer is computed by:
        # (kernel_size - 1) * frame_rate_of_this_layer
        self.subsampling_rate = 4
        # 6 = (3 - 1) * 1 + (3 - 1) * 2
        self.right_context = 6

    def forward(
        self,
        x: torch.Tensor,
        x_mask: torch.Tensor,
        offset: Union[int, torch.Tensor] = 0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 4.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 4.
            torch.Tensor: positional encoding

        """
        x = x.unsqueeze(1)  # (b, c=1, t, f)
        x = self.conv(x)
        b, c, t, f = x.size()
        x = self.out(x.transpose(1, 2).contiguous().view(b, t, c * f))
        x, pos_emb = self.pos_enc(x, offset)
        return x, pos_emb, x_mask[:, :, 2::2][:, :, 2::2]


class Conv2dSubsampling6(BaseSubsampling):
    """Convolutional 2D subsampling (to 1/6 length).
    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.
        pos_enc (torch.nn.Module): Custom position encoding layer.
    """

    def __init__(self, idim: int, odim: int, dropout_rate: float,
                 pos_enc_class: torch.nn.Module):
        """Construct an Conv2dSubsampling6 object."""
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv2d(1, odim, 3, 2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(odim, odim, 5, 3),
            torch.nn.ReLU(),
        )
        self.linear = torch.nn.Linear(odim * (((idim - 1) // 2 - 2) // 3),
                                      odim)
        self.pos_enc = pos_enc_class
        # 10 = (3 - 1) * 1 + (5 - 1) * 2
        self.subsampling_rate = 6
        self.right_context = 10

    def forward(
        self,
        x: torch.Tensor,
        x_mask: torch.Tensor,
        offset: Union[int, torch.Tensor] = 0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Subsample x.
        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 6.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 6.
            torch.Tensor: positional encoding
        """
        x = x.unsqueeze(1)  # (b, c, t, f)
        x = self.conv(x)
        b, c, t, f = x.size()
        x = self.linear(x.transpose(1, 2).contiguous().view(b, t, c * f))
        x, pos_emb = self.pos_enc(x, offset)
        return x, pos_emb, x_mask[:, :, 2::2][:, :, 4::3]


class Conv2dSubsampling8(BaseSubsampling):
    """Convolutional 2D subsampling (to 1/8 length).

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

    def __init__(self, idim: int, odim: int, dropout_rate: float,
                 pos_enc_class: torch.nn.Module):
        """Construct an Conv2dSubsampling8 object."""
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv2d(1, odim, 3, 2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(odim, odim, 3, 2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(odim, odim, 3, 2),
            torch.nn.ReLU(),
        )
        self.linear = torch.nn.Linear(
            odim * ((((idim - 1) // 2 - 1) // 2 - 1) // 2), odim)
        self.pos_enc = pos_enc_class
        self.subsampling_rate = 8
        # 14 = (3 - 1) * 1 + (3 - 1) * 2 + (3 - 1) * 4
        self.right_context = 14

    def forward(
        self,
        x: torch.Tensor,
        x_mask: torch.Tensor,
        offset: Union[int, torch.Tensor] = 0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Subsample x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: Subsampled tensor (#batch, time', odim),
                where time' = time // 8.
            torch.Tensor: Subsampled mask (#batch, 1, time'),
                where time' = time // 8.
            torch.Tensor: positional encoding
        """
        x = x.unsqueeze(1)  # (b, c, t, f)
        x = self.conv(x)
        b, c, t, f = x.size()
        x = self.linear(x.transpose(1, 2).contiguous().view(b, t, c * f))
        x, pos_emb = self.pos_enc(x, offset)
        return x, pos_emb, x_mask[:, :, 2::2][:, :, 2::2][:, :, 2::2]


class LegacyLinearNoSubsampling(BaseSubsampling):
    """Linear transform the input without subsampling

    Args:
        idim (int): Input dimension.
        odim (int): Output dimension.
        dropout_rate (float): Dropout rate.

    """

    def __init__(self, idim: int, odim: int, dropout_rate: float,
                 pos_enc_class: torch.nn.Module):
        """Construct an linear object."""
        super().__init__()
        self.out = torch.nn.Sequential(
            torch.nn.Linear(idim, odim),
            torch.nn.LayerNorm(odim, eps=1e-5),
            torch.nn.Dropout(dropout_rate),
            torch.nn.ReLU(),
        )
        self.pos_enc = pos_enc_class
        self.right_context = 0
        self.subsampling_rate = 1

    def forward(
        self,
        x: torch.Tensor,
        x_mask: torch.Tensor,
        offset: Union[int, torch.Tensor] = 0
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Input x.

        Args:
            x (torch.Tensor): Input tensor (#batch, time, idim).
            x_mask (torch.Tensor): Input mask (#batch, 1, time).

        Returns:
            torch.Tensor: linear input tensor (#batch, time', odim),
                where time' = time .
            torch.Tensor: linear input mask (#batch, 1, time'),
                where time' = time .

        """
        x = self.out(x)
        x, pos_emb = self.pos_enc(x, offset)
        return x, pos_emb, x_mask

In [ ]:
class Upsample1D(nn.Module):
  def __init__(self,channels:int,out_channels:int,stride:int=2):
    super().__init__()
    self.channels=channels
    self.out_channels=out_channels
    self.stride=stride
    self.conv=nn.Conv1d(self.channels,self.out_channels,stride*2+1,stride=1,padding=0)

  def forward(self, inputs: torch.Tensor, input_lengths: torch.Tensor, conv_cache: torch.Tensor = torch.zeros(0, 0, 0)) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    outputs = F.interpolate(inputs, scale_factor=float(self.stride), mode="nearest")
    if conv_cache.size(2)==0:
      outputs=F.pad(outputs,(self.stride*2.0),value=0.0)
    else:
      assert conv_cache.size(2)==self.stride*2
    conv_cache_new=outputs[:,:,-self.stride*2]
    outputs=self.conv(outputs)
    return outputs,input_lengths*self.stride,conv_cache_new


class PreLookaheadLayer(nn.Module):
  def __init__(self,channels:int, pre_lookahead_len:int=1):
    super().__init__()
    self.channels=channels
    self.pre_lookahead_len=pre_lookahead_len
    self.conv1=nn.Conv1d(
        channels,channels,
        kernel_size=pre_lookahead_len+1,
        stride=1,
        padding=0,
    )
    self.conv2=nn.Conv1d(
        channels,channels,
        kernel_size=3,stride=1,padding=0,
    )

  def forward(self, inputs: torch.Tensor, context: torch.Tensor = torch.zeros(0, 0, 0), conv2_cache: torch.Tensor = torch.zeros(0, 0, 0)) -> Tuple[torch.Tensor, torch.Tensor]:
    outputs=inputs.transpose(1,2).contiguous()
    context=context.transpose(1,2).contiguous()

    if context.size(2)==0:
      outputs=F.pad(outputs,(0,self.pre_lookahead_len),mode='constant',value=0.0)
    else:
      assert context.size(2)==self.pre_lookahead_len
      outputs=F.pad(torch.concat([outputs,context],dim=2),(0,self.prelookahead_len-context.size(2)),mode='constant',value=0.0)
    outputs=F.leaky_relu(self.conv1(outputs))

    if conv2_cache.size(2)==0:
      outputs=F.pad(outputs,(self.conv2.kernel_size[0]-1,0),mode='constant',value=0.0)
    else:
      assert conv2_cache.size(2)==self.conv2.kernel_size[0]-1
      outputs=torch.concat([conv2_cache,outputs],dim=2)
    conv2_cache_new=outputs[:,:,-(self.conv2.kernel_size[0]-1):]
    outputs=self.conv2(outputs)
    outputs=outputs.transpose(1,2).contiguous()

    outputs=outputs+inputs
    return outputs, conv2_cache_new

In [ ]:
class UpsampleConformerEncoder(torch.nn.Module):

    def __init__(
        self,
        input_size: int,
        output_size: int = 256,
        attention_heads: int = 4,
        linear_units: int = 2048,
        num_blocks: int = 6,
        dropout_rate: float = 0.1,
        positional_dropout_rate: float = 0.1,
        attention_dropout_rate: float = 0.0,
        input_layer: str = "conv2d",
        pos_enc_layer_type: str = "rel_pos",
        normalize_before: bool = True,
        static_chunk_size: int = 0,
        use_dynamic_chunk: bool = False,
        global_cmvn: torch.nn.Module = None,
        use_dynamic_left_chunk: bool = False,
        positionwise_conv_kernel_size: int = 1,
        macaron_style: bool = True,
        selfattention_layer_type: str = "rel_selfattn",
        activation_type: str = "swish",
        use_cnn_module: bool = True,
        cnn_module_kernel: int = 15,
        causal: bool = False,
        cnn_module_norm: str = "batch_norm",
        key_bias: bool = True,
        gradient_checkpointing: bool = False,
    ):
        """
        Args:
            input_size (int): input dim
            output_size (int): dimension of attention
            attention_heads (int): the number of heads of multi head attention
            linear_units (int): the hidden units number of position-wise feed
                forward
            num_blocks (int): the number of decoder blocks
            dropout_rate (float): dropout rate
            attention_dropout_rate (float): dropout rate in attention
            positional_dropout_rate (float): dropout rate after adding
                positional encoding
            input_layer (str): input layer type.
                optional [linear, conv2d, conv2d6, conv2d8]
            pos_enc_layer_type (str): Encoder positional encoding layer type.
                opitonal [abs_pos, scaled_abs_pos, rel_pos, no_pos]
            normalize_before (bool):
                True: use layer_norm before each sub-block of a layer.
                False: use layer_norm after each sub-block of a layer.
            static_chunk_size (int): chunk size for static chunk training and
                decoding
            use_dynamic_chunk (bool): whether use dynamic chunk size for
                training or not, You can only use fixed chunk(chunk_size > 0)
                or dyanmic chunk size(use_dynamic_chunk = True)
            global_cmvn (Optional[torch.nn.Module]): Optional GlobalCMVN module
            use_dynamic_left_chunk (bool): whether use dynamic left chunk in
                dynamic chunk training
            key_bias: whether use bias in attention.linear_k, False for whisper models.
            gradient_checkpointing: rerunning a forward-pass segment for each
                checkpointed segment during backward.
        """
        super().__init__()
        self._output_size = output_size

        self.global_cmvn = global_cmvn
        self.embed = COSYVOICE_SUBSAMPLE_CLASSES[input_layer](
            input_size,
            output_size,
            dropout_rate,
            COSYVOICE_EMB_CLASSES[pos_enc_layer_type](output_size,
                                                      positional_dropout_rate),
        )

        self.normalize_before = normalize_before
        self.after_norm = torch.nn.LayerNorm(output_size, eps=1e-5)
        self.static_chunk_size = static_chunk_size
        self.use_dynamic_chunk = use_dynamic_chunk
        self.use_dynamic_left_chunk = use_dynamic_left_chunk
        self.gradient_checkpointing = gradient_checkpointing
        activation = COSYVOICE_ACTIVATION_CLASSES[activation_type]()
        # self-attention module definition
        encoder_selfattn_layer_args = (
            attention_heads,
            output_size,
            attention_dropout_rate,
            key_bias,
        )
        # feed-forward module definition
        positionwise_layer_args = (
            output_size,
            linear_units,
            dropout_rate,
            activation,
        )
        # convolution module definition
        convolution_layer_args = (output_size, cnn_module_kernel, activation,
                                  cnn_module_norm, causal)
        self.pre_lookahead_layer = PreLookaheadLayer(channels=512, pre_lookahead_len=3)
        self.encoders = torch.nn.ModuleList([
            ConformerEncoderLayer(
                output_size,
                COSYVOICE_ATTENTION_CLASSES[selfattention_layer_type](
                    *encoder_selfattn_layer_args),
                PositionwiseFeedForward(*positionwise_layer_args),
                PositionwiseFeedForward(
                    *positionwise_layer_args) if macaron_style else None,
                ConvolutionModule(
                    *convolution_layer_args) if use_cnn_module else None,
                dropout_rate,
                normalize_before,
            ) for _ in range(num_blocks)
        ])
        self.up_layer = Upsample1D(channels=512, out_channels=512, stride=2)
        self.up_embed = COSYVOICE_SUBSAMPLE_CLASSES[input_layer](
            input_size,
            output_size,
            dropout_rate,
            COSYVOICE_EMB_CLASSES[pos_enc_layer_type](output_size,
                                                      positional_dropout_rate),
        )
        self.up_encoders = torch.nn.ModuleList([
            ConformerEncoderLayer(
                output_size,
                COSYVOICE_ATTENTION_CLASSES[selfattention_layer_type](
                    *encoder_selfattn_layer_args),
                PositionwiseFeedForward(*positionwise_layer_args),
                PositionwiseFeedForward(
                    *positionwise_layer_args) if macaron_style else None,
                ConvolutionModule(
                    *convolution_layer_args) if use_cnn_module else None,
                dropout_rate,
                normalize_before,
            ) for _ in range(4)
        ])

    def output_size(self) -> int:
        return self._output_size

    def forward(
        self,
        xs: torch.Tensor,
        xs_lens: torch.Tensor,
        decoding_chunk_size: int = 0,
        num_decoding_left_chunks: int = -1,
        streaming: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Embed positions in tensor.

        Args:
            xs: padded input tensor (B, T, D)
            xs_lens: input length (B)
            decoding_chunk_size: decoding chunk size for dynamic chunk
                0: default for training, use random dynamic chunk.
                <0: for decoding, use full chunk.
                >0: for decoding, use fixed chunk size as set.
            num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
                >=0: use num_decoding_left_chunks
                <0: use all left chunks
        Returns:
            encoder output tensor xs, and subsampled masks
            xs: padded output tensor (B, T' ~= T/subsample_rate, D)
            masks: torch.Tensor batch padding mask after subsample
                (B, 1, T' ~= T/subsample_rate)
        NOTE(xcsong):
            We pass the `__call__` method of the modules instead of `forward` to the
            checkpointing API because `__call__` attaches all the hooks of the module.
            https://discuss.pytorch.org/t/any-different-between-model-input-and-model-forward-input/3690/2
        """
        T = xs.size(1)
        masks = ~make_pad_mask(xs_lens, T).unsqueeze(1)  # (B, 1, T)
        if self.global_cmvn is not None:
            xs = self.global_cmvn(xs)
        xs, pos_emb, masks = self.embed(xs, masks)
        mask_pad = masks  # (B, 1, T/subsample_rate)
        chunk_masks = add_optional_chunk_mask(xs, masks, False, False, 0, self.static_chunk_size if streaming is True else 0, -1)
        # lookahead + conformer encoder
        xs, _ = self.pre_lookahead_layer(xs)
        xs = self.forward_layers(xs, chunk_masks, pos_emb, mask_pad)

        # upsample + conformer encoder
        xs = xs.transpose(1, 2).contiguous()
        xs, xs_lens, _ = self.up_layer(xs, xs_lens)
        xs = xs.transpose(1, 2).contiguous()
        T = xs.size(1)
        masks = ~make_pad_mask(xs_lens, T).unsqueeze(1)  # (B, 1, T)
        xs, pos_emb, masks = self.up_embed(xs, masks)
        mask_pad = masks  # (B, 1, T/subsample_rate)
        chunk_masks = add_optional_chunk_mask(xs, masks, False, False, 0, self.static_chunk_size * self.up_layer.stride if streaming is True else 0, -1)
        xs = self.forward_up_layers(xs, chunk_masks, pos_emb, mask_pad)

        if self.normalize_before:
            xs = self.after_norm(xs)
        # Here we assume the mask is not changed in encoder layers, so just
        # return the masks before encoder layers, and the masks will be used
        # for cross attention with decoder later
        return xs, masks

    def forward_layers(self, xs: torch.Tensor, chunk_masks: torch.Tensor,
                       pos_emb: torch.Tensor,
                       mask_pad: torch.Tensor) -> torch.Tensor:
        for layer in self.encoders:
            xs, chunk_masks, _, _ = layer(xs, chunk_masks, pos_emb, mask_pad)
        return xs

    def forward_up_layers(self, xs: torch.Tensor, chunk_masks: torch.Tensor,
                          pos_emb: torch.Tensor,
                          mask_pad: torch.Tensor) -> torch.Tensor:
        for layer in self.up_encoders:
            xs, chunk_masks, _, _ = layer(xs, chunk_masks, pos_emb, mask_pad)
        return xs

    @torch.jit.export
    def forward_chunk(
        self,
        xs: torch.Tensor,
        xs_lens: torch.Tensor,
        offset: int = 0,
        context: torch.Tensor = torch.zeros(0, 0, 0),
        pre_lookahead_layer_conv2_cache: torch.Tensor = torch.zeros(0, 0, 0),
        encoders_kv_cache: torch.Tensor = torch.zeros(0, 0, 0, 0, 0),
        upsample_offset: int = 0,
        upsample_conv_cache: torch.Tensor = torch.zeros(0, 0, 0),
        upsample_kv_cache: torch.Tensor = torch.zeros(0, 0, 0, 0, 0)
    ) -> Tuple[torch.Tensor, torch.Tensor, Tuple[int, torch.Tensor, torch.Tensor, int, torch.Tensor, torch.Tensor]]:
        """Embed positions in tensor.

        Args:
            xs: padded input tensor (B, T, D)
            xs_lens: input length (B)
            decoding_chunk_size: decoding chunk size for dynamic chunk
                0: default for training, use random dynamic chunk.
                <0: for decoding, use full chunk.
                >0: for decoding, use fixed chunk size as set.
            num_decoding_left_chunks: number of left chunks, this is for decoding,
            the chunk size is decoding_chunk_size.
                >=0: use num_decoding_left_chunks
                <0: use all left chunks
        Returns:
            encoder output tensor xs, and subsampled masks
            xs: padded output tensor (B, T' ~= T/subsample_rate, D)
            masks: torch.Tensor batch padding mask after subsample
                (B, 1, T' ~= T/subsample_rate)
        NOTE(xcsong):
            We pass the `__call__` method of the modules instead of `forward` to the
            checkpointing API because `__call__` attaches all the hooks of the module.
            https://discuss.pytorch.org/t/any-different-between-model-input-and-model-forward-input/3690/2
        """
        assert xs.size(0) == 1
        # tmp_masks is just for interface compatibility
        tmp_masks = torch.ones(1,
                               xs.size(1),
                               device=xs.device,
                               dtype=torch.bool)
        tmp_masks = tmp_masks.unsqueeze(1)
        if self.global_cmvn is not None:
            xs = self.global_cmvn(xs)
        # NOTE(xcsong): Before embed, shape(xs) is (b=1, time, mel-dim)
        xs, pos_emb, _ = self.embed(xs, tmp_masks, offset)
        offset += xs.size(1)
        tmp_masks = torch.ones(1,
                               context.size(1),
                               device=context.device,
                               dtype=torch.bool)
        tmp_masks = tmp_masks.unsqueeze(1)
        if context.size(1) != 0:
            context, _, _ = self.embed(context, tmp_masks, offset)

        # lookahead + conformer encoder
        xs, pre_lookahead_layer_conv2_cache = self.pre_lookahead_layer(xs, context, pre_lookahead_layer_conv2_cache)
        # NOTE in cache mode we do not need to call add_optional_chunk_mask
        chunk_masks = torch.ones((1, xs.size(1), offset), dtype=torch.bool, device=xs.device)
        mask_pad = torch.ones((0, 0, 0), dtype=torch.bool, device=xs.device)
        encoders_kv_cache_list = []
        for index, layer in enumerate(self.encoders):
            xs, chunk_masks, encoders_kv_cache_new, _ = layer(xs, chunk_masks, pos_emb, mask_pad, encoders_kv_cache[index])
            encoders_kv_cache_list.append(encoders_kv_cache_new)
        encoders_kv_cache = torch.stack(encoders_kv_cache_list, dim=0)

        # upsample
        xs = xs.transpose(1, 2).contiguous()
        xs, xs_lens, upsample_conv_cache = self.up_layer(xs, xs_lens, upsample_conv_cache)
        xs = xs.transpose(1, 2).contiguous()

        # tmp_masks is just for interface compatibility
        tmp_masks = torch.ones(1,
                               xs.size(1),
                               device=xs.device,
                               dtype=torch.bool)
        tmp_masks = tmp_masks.unsqueeze(1)
        xs, pos_emb, masks = self.up_embed(xs, tmp_masks, upsample_offset)
        upsample_offset += xs.size(1)

        # conformer encoder
        chunk_masks = torch.ones((1, xs.size(1), upsample_offset), dtype=torch.bool, device=xs.device)
        mask_pad = torch.ones((0, 0, 0), dtype=torch.bool, device=xs.device)
        upsample_kv_cache_list = []
        for index, layer in enumerate(self.up_encoders):
            xs, chunk_masks, upsample_kv_cache_new, _ = layer(xs, chunk_masks, pos_emb, mask_pad, upsample_kv_cache[index])
            upsample_kv_cache_list.append(upsample_kv_cache_new)
        upsample_kv_cache = torch.stack(upsample_kv_cache_list, dim=0)

        if self.normalize_before:
            xs = self.after_norm(xs)
        # Here we assume the mask is not changed in encoder layers, so just
        # return the masks before encoder layers, and the masks will be used
        # for cross attention with decoder later
        return xs, masks, (offset, pre_lookahead_layer_conv2_cache, encoders_kv_cache, upsample_offset, upsample_conv_cache, upsample_kv_cache)

In [ ]:
class BaseEncoder(torch.nn.Module):
  def __init__(
      self,
      input_size:int,
      output_size:int=256,
      attention_heads:int=4,
      linear_units:int=2058,
      num_blocks:int=6,
      dropout_rate:float=0.1,
      positional_dropout_rate:float=0.1,
      attention_dropout_rate:float=0.0,
      input_layer:str="conv2d",
      pos_enc_layer_type:str="abs_pos",
      normalize_before:bool=True,
      static_chunk_size:int=0,
      use_dynamic_chunk:bool=False,
      global_cmvn:torch.nn.Module=None,
      use_dynamic_left_chunk:bool=False,
      gradient_checkpointing:bool=False,

  ):

    super().__init__()
    self._output_size=output_size

    self.global_cmvn=global_cmvn
    self.embed_COSYVOICE_SUBSAMPLE_CLASSES[input_layer](
        input_size,
        output_size,
        dropout_rate,
        COSYVOICE_EMB_CLASSES[pos_enc_layer_type](output_size,positional_dropout_rate),
    )

    self.normalize_before=normalize_before
    self.after_norm=torch.nn.LayerNorm(output_size,eps=1e-5)
    self.static_chunk_size=static_chunk_size
    self.use_dynamic_chunk=use_dynamic_chunk
    self.use_dynamic_left_chunk=use_dynamic_left_chunk
    self.gradient_checkpointing=gradient_checkpointing

  def output_size(self)->int:
    return self._output_size

  def forward(
      self,
      xs:torch.Tensor,
      xs_lens:torch.Tensor,
      decoding_chunk_size:int=0,
      num_decoding_left_chunks:int=-1,
  )->Tuple[torch.Tensor,torch.Tensor]:

  T=xs.size(1)
  masks=~make_pad_mask(xs_lens,T).unsqueeze(1)
  if self.global_cmvn is not None:
    xs=self.global_cmvn(xs)
    xs,pos_emb,masks=self.embed(xs,masks,
                                self.use_dynamic_chunk,
                                self.use_dynamic_left_chunk,
                                decoding_chunk_size,
                                self.static_chunk_size,
                                num_decoding_left_chunks)
    if self.gradient_checkpointing and self.training:
      xs=self.forward_layers_checkpointing(xs,chunk_masks,pos_emb,mask_pad)
    else:
      xs=self.forward_layers(xs,chunk_masks,pos_emb,mask_pad)
    if self.normalize_before:
      xs=self.after_norm(xs)

    return xs, masks

  def forward_chunk(
      self,
      xs:torch.Tensoro,
      offset:int,
      required_cache_size:int,
      att_cache:torch.Tensor=torch.zeros(0,0,0,0),
      cnn_cache:torch.Tensor=torch.zeros(0,0,0,0),
      att_mask:torch.Tensor=torch.zeros(0,0,0,0),

  )->Tuple[torch.Tensor,torch.Tensor,torch.Tensor]:

    assert xs.size(0)==1
    tmp_masks=torch.ones(1,
                         xs.size(1),
                         device=xs.device,
                         dtype=torch.bool)
    tmp_masks=tmp_masks.unsqueeze(1)
    if self.global_cmvn is not None:
      xs=self.global_cmvn(xs)
      xs,pos_emb,_=self.embed(xs,tmp_masks,offset)
      elayers,cache_t1=att_cache.size(0),att_cache.size(2)
      chunk_size=xs.size(1)
      attention_key_size=cache_t1+chunk_size
      pos_emb=self.embed.position_encoding(offset=offset-cache_t1,
                                           size=attention_key_size)
      if required_cache_size<0:
        next_cache_start=0
      elif required_cache_size==0:
        next_cache_start=attention_key_size
      else:
        next_cache_start=max(attention_key_size-required_cache_size,0)
      r_att_cache=[]
      r_cnn_cache=[]
      for i, layer in enumerate(self.encoders):
        xs,_,new_att_cache, new_cnn_cache=layer(
            xs,
            att_mask,
            pos_emb,
            att_cache=att_cache[i:i+1] if elayers>0 else att_cache,
            cnn_cache=cnn_cache[i] if cnn_cache.size(0)>0 else cnn_cache)
        r_att_cache.append(new_att_cache[:,:,next_cache_start:,:])
        r_cnn_cache.append(new_cnn_cache.unsqueeze(0))
      if self.normalize_before:
        xs=self.after_norm(xs)

      r_att_cache=torch.cat(r_att_cache,dim=0)
      r_cnn_cache=torch.cat(r_cnn_cache,dim=0)

      return (xs,r_att_cache,r_cnn_cache)


  def forward_chunk_by_chunk(
      self,
      xs:torch.Tensor,
      decoding_chunk_size:int,
      num_decoding_left_chunks:int=-1,
      )->Tuple[torch.Tensor,torch.Tensor]:

    assert decoding_chunk_size>0
    assert self.static_chunk_size>0 or self.use_dynamic_chunk
    subsampling=self.embed.subsampling_rate
    context=self.embed.right_context+1
    stride=subsampling*decoding_chunk_size
    decoding_window=(decoding_chunk_size-1)*subsampling+context
    num_frames=xs.size(1)
    att_cache:torch.Tensor=torch.zeros((0,0,0,0),device=xs.device)
    cnn_cache:torch.Tensor=torch.zeros((0,0,0,0),device=xs.device)
    outputs=[]
    offset=0
    required_cache_size=decoding_chunk_size*num_decoding_left_chunks

    for cur in range(0,num_frams-context+1,stride):
      end=min(cur+decoding_window,num_frames)
      chunk_xs=xs[:,cur:end,:]
      (y,att_cache,cnn_cache)=self.forward_chunk(chunk_xs,offset,required_cache_size,att_cache,cnn_cache)
      outputs.append(y)
      offset+=y.size(1)
    ys=torch.cat(outputs,1)
    masks=torch.ones((1,1,ys.size(1)),
                      device=ys.device,
                      dtype=torch.bool)
    return ys,masks



In [1]:
class TransformerEncoder(BaseEncoder):

  def __init__(
      self,
      input_size:int,
      output_size:int=256,
      attention_heads:int=4,
      linear_units:int=2048,
      num_blocks:int=6,
      dropout_rate:float=0.1,
      attention_dropout_rate:float=0.0,
      positional_dropout_rate:float=0.1,
      input_layer:str="conv2d",
      pos_enc_layer_type:str="abs_pos",
      normalize_before:bool=True,
      static_chunk_size:int=0,
      use_dynamic_chunk:bool=False,
      global_cmvn:torch.nn.Module=None,
      use_dynamic_left_chunk:bool=False,
      key_bias:bool=True,
      selfattention_layer_type:str="selfattn",
      activation_type:str="relu",
      gradient_checkpointing:bool=False,
  ):

    super().__init__(input_size, output_size, attention_heads,
                         linear_units, num_blocks, dropout_rate,
                         positional_dropout_rate, attention_dropout_rate,
                         input_layer, pos_enc_layer_type, normalize_before,
                         static_chunk_size, use_dynamic_chunk, global_cmvn,
                         use_dynamic_left_chunk, gradient_checkpointing)
    activation = COSYVOICE_ACTIVATION_CLASSES[activation_type]()
    self.encoders=torch.nn.ModuleList([
        TransformerEncoderLayer(
                output_size,
                COSYVOICE_ATTENTION_CLASSES[selfattention_layer_type](attention_heads,
                                                                      output_size,
                                                                      attention_dropout_rate,
                                                                      key_bias),
                PositionwiseFeedForward(output_size, linear_units,
                                        dropout_rate, activation),
                dropout_rate, normalize_before) for _ in range(num_blocks)
        ])


class ConformerEncoder(BaseEncoder):
  def __init__(
      self,
      input_size:int,
      output_size:int=256,
      attention_heads:int=4,
      linear_units:int=2048,
      num_blocks:int=6,
      dropout_rate:float=0.1,
      positional_dropout_rate:float=0.1,
      attention_dropout_rate:float-0.0,
      input_layer:str="conv2d",
      pos_enc_layer_type:str="rel_pos",
      normalize_before:bool=True,
      static_chunk_size:int=0,
      use_dynamic_chunk:bool=False,
      global_cmvn:torch.nn.Module=None,
      use_dynamic_left_chunk:bool=False,
      positionwise_conv_kernel_size:int=1,
      macaron_style:bool=True,
      selfattention_layer_type:str="rel_selfattn",
      activation_type:str="swish",
      use_cnn_module:bool=True,
      cnn_module_kernel:int=15,
      casual:bool=False,
      cnn_module_norm:str="batch_norm",
      key_bias:bool=True,
      gradient_checkpointing:bool=False,
  )
    super().__init__(input_size, output_size, attention_heads,
                          linear_units, num_blocks, dropout_rate,
                          positional_dropout_rate, attention_dropout_rate,
                          input_layer, pos_enc_layer_type, normalize_before,
                          static_chunk_size, use_dynamic_chunk, global_cmvn,
                          use_dynamic_left_chunk, gradient_checkpointing)
          activation = COSYVOICE_ACTIVATION_CLASSES[activation_type]()
    encoder_selfattn_layer_args=(
        attention_heads,
        output_size,
        attention_dropout_rate,
        key_bias,
    )
    positionwise_layer_args=(
        output_size,
        linear_units,
        dropout_rate,
        activation,
    )
    convolution_layer_args=(output_size,cnn_module_kernel,activation,
                            cnn_module_norm,causal)

    self.encoders=torch.nn.ModuleList([
        ConformerEncoderLayer(
            output_size,
            COSYVOICE_ATTENTION_CLASSES[selfattention_layer_type](
                *encoder_selfattn_layer_args
            ),
            PositionwiseFeedForward(*positionwise_layer_args),
            PositionwiseFeedForward(*positionwise_layer_args) if macaron_style else None,
            ConvolutionModule(*convolution_layer_args) if use_cnn_module else None,
            dropout_rate,normalize_before,
        ) for _ in range(num_blocks)
    ])


SyntaxError: non-default argument follows default argument (<ipython-input-1-e4cfe7f2bd50>, line 51)

In [1]:
class TransformerDecoder(torch.nn.Module):
  def __init__(
      self,
      vocab_size:int,
      encoder_output_size:int,
      attention_heads:int=4,
      linear_units:int=2048,
      num_blocks:int=6,
      dropout_rate:float=0.1,
      positional_dropout_rate:float=0.1,
      self_attention_dropout_rate:float=0.0,
      src_attention_dropout_rate:float=0.0,
      input_layer:str="conv2d",
      use_output_layer:bool=True,
      normalize_before:bool=True,
      src_attention:bool=True,
      key_bias:bool=True,
      activation_type:str="relu",
      gradient_checkpointing:bool=False,
      tie_word_embedding:bool=False,
  ):
    super().__init__()
    attention_dim=encodeer_output_size
    activation=COSYVOICE_ACTIVATION_CLASSES[activation_type]()

    self.embed=torch.nn.Sequential(
        torch.nn.Identity() if input_layer=="no_pos" else
        torch.nn.Embedding(vocab_size,attention_dim),
        COSYVOICE_EMB_CLASSES[input_layer](attention_dim,
                                           positional_dropout_rate),
    )

    self.normalize_before=normalize_before
    self.after_norm=torch.nn.LayerNorm(attention_dim,eps=1e-5)
    self.use_output_layer=use_output_layer
    if use_output_layer:
      self.output_layer=torch.nn.Linear(attention_dim,vocab_size)
    else:
      self.output_layer=torch.nn.Identity()
    self.num_blocks=num_blocks
    self.decoders=torch.nn.ModuleList([
        DecoderLayer(
            attention_dim,
            COSYVOICE_ATTENTION_CLASSES["selfattn"](
                attention_heads,attention_dim,
                self_attention_dropout_rate,key_bias),
            COSYVOICE_ATTENTION_CLASSES["selfattn"](
                attention_heads,attention_dim,src_attention_dropout_rate,
                key_bias) if src_attention else None,
            PositionwiseFeedForward(attention_dim,linear_units,
                                    dropout_rate,activation),
            dropout_rate,
            normalize_before,
        ) for _ in range(self.num_blocks)
    ])
    self.gradient_checkpointing=gradient_checkpointing
    self.tie_word_embedding=tie_word_embedding


  def forward(
      self,
      memory:torch.Tensor,
      memory_mask:torch.Tensor,
      ys_in_pad:torch.Tensor,
      ys_in_lens:torch.Tensor,
      r_ys_in_pad:torch.Tensor,
      reverse_weight:float=0.0,
  ) -> Tuple[torch.Tensor,torch.Tensor,torch.Tensor]:

    tgt=ys_in_pad
    maxlen=tgt.size(1)
    tgt_mask=~mask_pad_mask(ys_in_lens,maxlen).unsqueeze(1)
    tgt_mask=tgt_mask.to(tgt.device)
    m=subsequent_mask(tgt_mask.size(-1),
                      device=tgt_mask.device).unsqueeze(0)
    tgt_mask=tgt_mask&m
    x,_=self.embed(tgt)
    if self.gradient_checkpointing and self.training:
      x=self.forward_layers_checkpointed(x,tgt_mask,memory,memory_mask)

    else:
      x=self.forward_layers(x,tgt_mask,memory,memory_mask)

    if self.normalize_before:
      x=self.after_norm(x)
    if self.use_output_layer:
      x=self.output_layer(x)
    olens=tgt_mask.sum(1)
    return x, torch.tensor(0.0),olens

  def forward_layers(self,x:torch.Tensor,tgt_mask:torch.Tensor,
                     memory:torch.Tensor,
                     memory_mask:torch.Tensor)->torch.Tensor:
    for layer in self.decoders:
      x,tgt_mask,memory,memory_mask=layer(x,tgt_mask,memory,memory_mask)
    return x


  @torch.jit.unused
  def forward_layers_checkpointed(self,x:torch.Tensor,
                                  tgt_mask:torch.Tensor,
                                  memory:torch.Tensor,
                                  memory_mask:torch.Tensor)->torch.Tensor:
    for layer in self.decoders:
      x,tgt_mask,memory,memory_mask=ckpt.checkpoint(
          layer.__call__,x,tgt_mask,memory,memory_mask)
    return x


  def forward_one_step(
      self,
      memory:torch.Tensor,
      memory_mask:torch.Tensor,
      tgt:torch.Tensor,
      tgt_mask:torch.Tensor,
      cache:Optional[List[torch.Tensor]]=None,
  )->Tuple[torch.Tensor,List[torch.Tensor]]:

    x,_=self.embed(tgt)
    new_cache=[]
    for i, decoder in enumerate(self.decoders):
      if cache is None:
        c=None
      else:
        c=cache[i]
      x,tgt_mask,memory,memory_mask=decoder(x,
                                            tgt_mask,
                                            memory,
                                            memory_mask,
                                            cache=c)
      new_cache.append(x)
      if self.normalize_before:
        y=self.after_norm(x[:,-1])
      else:
        y=x[:,-1]
      if self.use_output_layer:
        y=torch.log_softmax(self.output_layer(y),dim=-1)
      return y, new_cache

  def tie_or_clone_weights(self,jit_mode:bool=True):
    if not self.use_output_layer:
      return
    if jit_mode:
      logging.info("clone emb.weight to output.weight")
      self.output_layer.weight=torch.nn.Parameter(
          self.embed[0].weight.clone())
    else:
      logging.info("tie emb.weight with output.weight")
      self.output_layer.weight=self.embed[0].weight

    if getattr(self.output_layer,"bias",None) is not None:
      self.output_layer.bias.data=torch.nn.functional.pad(
          self.output_layer.bias.data,(
              0,
              self.output_layer.weight.shape[0]-
              self.output_layer.bias.shape[0],
          ),
          "constant",
          0,
      )

NameError: name 'torch' is not defined

In [ ]:
class BiTransformerDecoder(torch.nn.Module):
  def __init__(
      self,
      vocab_size:int,
      encoder_output_size:int,
      attention_heads:int=4,
      linear_units:int=2048,
      num_blocks:int=6,
      r_num_blocks:int=0,
      dropout_rate:float=0.1,
      positional_dropout_rate:float=0.1,
      self_attention_dropout_rate:float=0.0,
      src_attention_dropout_rate:float=0.0,
      input_layer:str="embed",
      use_output_layer:bool=True,
      normalize_before:bool=True,
      key_bias:bool=True,
      gradient_checkpointing:bool=False,
      tie_word_embedding:bool=False,
  ):

  def __init__(
      self,
      vocab_size:int,
      encoder_output_size:int,
      attention_heads:int=4,
      linear_units:int=2048,
      num_blocks:int=6,
      r_num_blocks:int=0,
      dropout_rate:float=0.1,
      positional_dropout_rate:float=0.1,
      self_attention_dropout_rate:float=0.0,
      src_attention_dropout_rate:float=0.0,
      input_layer:str="embed",
      use_output_layer:bool=True,
      normalize_before:bool=True,
      key_bias:bool=True,
      gradient_checkpointing:bool=False,
      tie_word_embedding:bool=False,
  ):
    super().__init__()
    self.tie_word_embedding=tie_word_embedding
    self.left_decoder = TransformerDecoder(
            vocab_size,
            encoder_output_size,
            attention_heads,
            linear_units,
            num_blocks,
            dropout_rate,
            positional_dropout_rate,
            self_attention_dropout_rate,
            src_attention_dropout_rate,
            input_layer,
            use_output_layer,
            normalize_before,
            key_bias=key_bias,
            gradient_checkpointing=gradient_checkpointing,
            tie_word_embedding=tie_word_embedding)

    self.right_decoder = TransformerDecoder(
            vocab_size,
            encoder_output_size,
            attention_heads,
            linear_units,
            r_num_blocks,
            dropout_rate,
            positional_dropout_rate,
            self_attention_dropout_rate,
            src_attention_dropout_rate,
            input_layer,
            use_output_layer,
            normalize_before,
            key_bias=key_bias,
            gradient_checkpointing=gradient_checkpointing,
            tie_word_embedding=tie_word_embedding)

  def forward(
      self,
      memory:torch.Tensor,
      memory_mask:torch.Tensor,
      ys_in_pad:torch.Tensor,
      ys_in_lens:torch.Tensor,
      r_ys_in_pad:torch.Tensor,
      reverse_weight:float=0.0,
      )->Tuple[torch.Tensor,torch.Tensor,torch.Tensor]:

    l_x,_,olens=self.left_decoder(memory,memory_mask,ys_in_pad,ys_in_lens)
    r_x=torch.tensor(0.0)
    if reverse_weight>0.0:
      r_x,_,olens=self.right_decoder(memory,memory_mask,r_ys_in_pad,ys_in_lens)
    return l_x,r_x,olens


  def forward_one_step(
      self,
      memory:torch.Tensor,
      memory_mask:torch.Tensor,
      tgt:torch.Tensor,
      tgt_mask:torch.Tensor,
      cache:Optional[List[torch.Tensor]]=None,
  )->Tuple[torch.Tensor,List[torch.Tensor]]:

    return self.left_decoder.forward_one_step(memory,memory_mask,tgt,tgt_mask,cache)

  def tie_or_clone_weights(self,jit_mode:bool=True):
    self.left_decoder.tie_or_clone_weights(jit_mode)
    self.right_decoder.tie_or_clone_weights(jit_mode)